# 5. Hypothesis testing

Use α = 0.05 and two-sided tests. Chi-square tests evaluate categorical association; Mann-Whitney U is a nonparametric comparison of two independent numeric distributions. Report effect sizes as well as p-values. The test results describe association only, not causation.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from src.data_loader import load_data
from src.stats_tests import run_hypothesis_tests

df = load_data()
results = run_hypothesis_tests(df)
display(results)

,hypothesis,test,statistic,degrees_freedom,p_value,effect_size_name,effect_size,sample_size,significant_at_0_05
0,Survival outcome is associated with passenger ...,Chi-square test of independence,260.717020,1.0,1.197357e-58,Cramer's V,0.542621,891,True
1,Survival outcome is associated with passenger ...,Chi-square test of independence,102.888989,2.0,4.549252e-23,Cramer's V,0.336684,891,True
2,Survivors and non-survivors have different far...,Mann-Whitney U (two-sided),129951.500000,NaN,4.553477e-22,Rank-biserial correlation,0.384245,891,True
3,Survivors and non-survivors have different age...,Mann-Whitney U (two-sided),57682.000000,NaN,1.604925e-01,Rank-biserial correlation,-0.061776,714,False


## Assumption checks

Check expected cell counts for the chi-square tests. For the continuous outcomes, Shapiro-Wilk and Levene checks are included as diagnostics; with this sample size and visibly skewed fares, the rank-based Mann-Whitney tests are used rather than assuming normality and equal variances. Independent passenger records remain a design assumption.

In [2]:
from scipy import stats
import pandas as pd

assumptions = []
for field in ["sex", "pclass"]:
    table = pd.crosstab(df[field], df["survived"])
    _, _, _, expected = stats.chi2_contingency(table)
    assumptions.append({"check": f"{field} x survived: minimum expected cell count",
                        "value": expected.min(), "threshold": 5,
                        "passes": bool((expected >= 5).all())})
for outcome in ["age", "fare"]:
    groups = [df.loc[df["survived"] == value, outcome].dropna() for value in [0, 1]]
    shapiro_p = [stats.shapiro(group).pvalue for group in groups]
    levene_p = stats.levene(*groups, center="median").pvalue
    assumptions.extend([
        {"check": f"{outcome}: Shapiro-Wilk p-values (non-normal if < 0.05)",
         "value": shapiro_p, "threshold": 0.05,
         "passes": all(p >= 0.05 for p in shapiro_p)},
        {"check": f"{outcome}: Levene p-value (unequal variance if < 0.05)",
         "value": levene_p, "threshold": 0.05,
         "passes": bool(levene_p >= 0.05)},
    ])
pd.DataFrame(assumptions)

,check,value,threshold,passes
0,sex x survived: minimum expected cell count,120.525253,5.00,True
1,pclass x survived: minimum expected cell count,70.626263,5.00,True
2,age: Shapiro-Wilk p-values (non-normal if < 0.05),"[7.81552733877256e-08, 0.0014264514374719827]",0.05,False
3,age: Levene p-value (unequal variance if < 0.05),0.274616,0.05,True
4,fare: Shapiro-Wilk p-values (non-normal if < 0...,"[3.2338412268801584e-36, 1.8337104339572837e-27]",0.05,False
5,fare: Levene p-value (unequal variance if < 0.05),0.0,0.05,False


Chi-square effect size is bias-corrected Cramer's V; Mann-Whitney effect size is rank-biserial correlation (positive values indicate the first listed group tends to have higher values). An assumption check for chi-square is that expected cell counts are adequate. Mann-Whitney does not require normally distributed outcomes, but independent observations are assumed and distribution-shape differences affect interpretation. Inspect the charts in notebook 3 alongside these results.

In [3]:
for row in results.to_dict(orient="records"):
    verdict = "evidence against the null at α=0.05" if row["significant_at_0_05"] else "insufficient evidence against the null at α=0.05"
    print(f"{row['hypothesis']} {verdict}; p={row['p_value']:.4g}, "
          f"{row['effect_size_name']}={row['effect_size']:.3f}.")

Survival outcome is associated with passenger sex. evidence against the null at α=0.05; p=1.197e-58, Cramer's V=0.543.
Survival outcome is associated with passenger class. evidence against the null at α=0.05; p=4.549e-23, Cramer's V=0.337.
Survivors and non-survivors have different fare distributions. evidence against the null at α=0.05; p=4.553e-22, Rank-biserial correlation=0.384.
Survivors and non-survivors have different age distributions. insufficient evidence against the null at α=0.05; p=0.1605, Rank-biserial correlation=-0.062.
